# CPDS-AI: Audio Classification (Baby Cry vs Car Noise)

This notebook is designed for Kaggle or Google Colab. The workflow includes:
1. Loading an audio dataset (Donate-a-Cry and ESC-50)
2. Extracting Mel-spectrogram features
3. Training a lightweight CNN (ResNet18)
4. Exporting an ONNX model.

In [ ]:
!pip install -q librosa onnx onnxruntime soundfile
import librosa
import torch
import torch.nn as nn
import torchvision.models as models
import json
from pathlib import Path
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split

## 0. Prepare Reproducible Training Data
The notebook clones Donate-a-cry and ESC-50 from GitHub automatically. It selects vehicle-relevant noise and explicitly excludes `crying_baby`.

In [ ]:
import csv
import shutil
import subprocess
import random
from pathlib import Path

SEED = 42
MAX_SAMPLES_PER_CLASS = 400
SOURCE_AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".ogg", ".caf", ".3gp", ".m4a"}
# The generated training set contains WAV only, avoiding decoder fallbacks during training.
AUDIO_EXTENSIONS = {".wav"}
VEHICLE_NOISE_CATEGORIES = {
    "engine", "car_horn", "siren", "rain", "wind", "airplane",
    "helicopter", "train", "clock_alarm", "washing_machine",
    "vacuum_cleaner", "door_wood_knock",
}
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Clone Donate-a-cry. Enable Internet in Kaggle before running this cell.
!if [ ! -d "donateacry-corpus" ]; then git clone --depth 1 https://github.com/gveres/donateacry-corpus.git; fi

# Attach Kaggle dataset: Environmental Sound Classification 50.
ESC50_ROOT = Path("/kaggle/input/environmental-sound-classification-50")
if not ESC50_ROOT.is_dir():
    raise FileNotFoundError("Attach the Environmental Sound Classification 50 Kaggle dataset as an Input.")

metadata_path = next(ESC50_ROOT.rglob("esc50.csv"), None)
audio_root = next((path for path in ESC50_ROOT.rglob("audio") if path.is_dir()), None)
if metadata_path is None or audio_root is None:
    raise FileNotFoundError("ESC-50 input must contain meta/esc50.csv and an audio directory.")

# Rebuild the generated dataset on every run so it is deterministic.
generated_root = Path("/kaggle/working/cpds-audio")
if generated_root.exists():
    shutil.rmtree(generated_root)
base_dir = generated_root / "train"
(base_dir / 'cry').mkdir(parents=True, exist_ok=True)
(base_dir / 'noise').mkdir(parents=True, exist_ok=True)

def copy_unique(source, destination, prefix):
    destination_path = destination / f"{prefix}_{source.name}"
    suffix = 1
    while destination_path.exists():
        destination_path = destination / f"{prefix}_{suffix}_{source.name}"
        suffix += 1
    shutil.copy2(source, destination_path)

def convert_to_wav(source, destination, prefix):
    destination_path = destination / f"{prefix}_{source.stem}.wav"
    suffix = 1
    while destination_path.exists():
        destination_path = destination / f"{prefix}_{suffix}_{source.stem}.wav"
        suffix += 1
    command = ["ffmpeg", "-nostdin", "-y", "-v", "error", "-i", str(source), "-ac", "1", "-ar", "16000", str(destination_path)]
    result = subprocess.run(command, capture_output=True, text=True)
    return destination_path if result.returncode == 0 and destination_path.is_file() else None

# Donate-a-cry stores tag information in filenames, so scan every supported audio file.
cry_source = Path("donateacry-corpus")
cry_files = sorted(path for path in cry_source.rglob("*") if path.suffix.lower() in SOURCE_AUDIO_EXTENSIONS)
if not cry_files:
    raise ValueError("Donate-a-cry did not provide any supported audio files.")
selected_cry = random.sample(cry_files, min(MAX_SAMPLES_PER_CLASS, len(cry_files)))
converted_cry = [convert_to_wav(audio_file, base_dir / "cry", "donateacry") for audio_file in selected_cry]
converted_cry = [path for path in converted_cry if path is not None]
if not converted_cry:
    raise RuntimeError("Unable to convert Donate-a-cry audio. Kaggle must provide ffmpeg.")

# Select only documented vehicle-relevant ESC-50 classes. The crying_baby class is never used as noise.
with metadata_path.open(newline="", encoding="utf-8") as csv_file:
    records = list(csv.DictReader(csv_file))
noise_records = [
    record for record in records
    if record["category"] in VEHICLE_NOISE_CATEGORIES and (audio_root / record["filename"]).is_file()
]
if not noise_records:
    raise ValueError("No selected ESC-50 vehicle-noise samples were found.")
selected_noise = random.sample(noise_records, min(len(converted_cry), len(noise_records)))
for record in selected_noise:
    copy_unique(audio_root / record["filename"], base_dir / "noise", f"esc50_{record['category']}")

manifest = {
    "seed": SEED,
    "cry_source": "https://github.com/gveres/donateacry-corpus.git",
    "esc50_input": "https://www.kaggle.com/datasets/mmoreaux/environmental-sound-classification-50",
    "excluded_categories": ["crying_baby"],
    "selected_noise_categories": sorted(VEHICLE_NOISE_CATEGORIES),
    "requested_cry_count": len(selected_cry),
    "cry_count": len(converted_cry),
    "noise_count": len(selected_noise),
}
(generated_root / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("✅ Datasets successfully downloaded and organized!")
print(json.dumps(manifest, indent=2))


## 1. Audio Preprocessing (Audio to Mel-Spectrogram)
The one-dimensional waveform is converted into a two-dimensional Mel-spectrogram.

In [ ]:
def extract_mel_spectrogram(audio_path, sr=16000, duration=2.0):
    """
    Load an audio file and create a fixed-size Mel-spectrogram.
    """
    y, sr = librosa.load(audio_path, sr=sr, duration=duration)
    
    # Ensure a fixed waveform length.
    target_length = int(sr * duration)
    if len(y) < target_length:
        y = np.pad(y, (0, target_length - len(y)))
    else:
        y = y[:target_length]
        
    # Compute the Mel-spectrogram.
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Normalize to [0, 1] and reshape for a one-channel CNN.
    mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-6)
    return np.expand_dims(mel_spec_db, axis=0)  # Shape: (1, 128, T)

## 2. Model Definition (Compact ResNet18)

In [ ]:
class AudioCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(AudioCNN, self).__init__()
        # Adapt ResNet18 to accept one channel instead of three RGB channels.
        self.model = models.resnet18(weights=None)
        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        # Replace the classifier with two classes: baby cry and noise.
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        return self.model(x)

print("Model definition is ready.")

## 3. Training

The dataset is pulled automatically from `/kaggle/working/cpds-audio`. The class order is saved next to the model so inference never guesses class indices.

In [ ]:
DATASET_DIR = Path("/kaggle/working/cpds-audio")
CLASS_NAMES = ["noise", "cry"]
SAMPLE_RATE, DURATION, BATCH_SIZE, EPOCHS = 16000, 2.0, 32, 20

class AudioDataset(Dataset):
    extensions = AUDIO_EXTENSIONS
    def __init__(self, root, class_names):
        self.samples = [(path, index) for index, label in enumerate(class_names)
                        for path in (root / label).rglob("*") if path.suffix.lower() in self.extensions]
        if not self.samples:
            raise ValueError(f"No audio files found under {root}")
    def __len__(self): return len(self.samples)
    def __getitem__(self, index):
        path, label = self.samples[index]
        return torch.from_numpy(extract_mel_spectrogram(path)).float(), label

train_root = DATASET_DIR / "train"
if not train_root.is_dir():
    raise FileNotFoundError(f"Set DATASET_DIR correctly; missing {train_root}")
train_dataset = AudioDataset(train_root, CLASS_NAMES)
val_root = DATASET_DIR / "val"
if val_root.is_dir():
    val_dataset = AudioDataset(val_root, CLASS_NAMES)
else:
    train_size = int(0.8 * len(train_dataset))
    train_dataset, val_dataset = random_split(train_dataset, [train_size, len(train_dataset) - train_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AudioCNN(num_classes=len(CLASS_NAMES)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
best_accuracy = -1.0

for epoch in range(EPOCHS):
    model.train()
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(inputs.to(device)), targets.to(device))
        loss.backward()
        optimizer.step()
    model.eval(); correct = total = 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            predictions = model(inputs.to(device)).argmax(dim=1).cpu()
            correct += (predictions == targets).sum().item(); total += len(targets)
    accuracy = correct / max(total, 1)
    print(f"Epoch {epoch + 1}/{EPOCHS}: validation accuracy={accuracy:.3f}")
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model.state_dict(), "audio_model.pth")

Path("audio_labels.json").write_text(json.dumps(CLASS_NAMES), encoding="utf-8")
print(f"Saved best model (validation accuracy {best_accuracy:.3f}) and labels.")

## 4. ONNX Export

In [ ]:
import shutil
import onnx
import onnxruntime as ort

# Load best weights.
model.load_state_dict(torch.load("audio_model.pth", map_location=device, weights_only=True))
model.eval()

# Example input for model tracing: (batch, channel, mels, time_steps).
dummy_input = torch.randn(1, 1, 128, 63)

# Export to ONNX.
torch.onnx.export(model,
                  dummy_input,
                  "audio_model.onnx",
                  export_params=True,
                  opset_version=11,
                  do_constant_folding=True,
                  input_names=['input'],
                  output_names=['output'],
                  dynamic_axes={'input': {0: 'batch_size', 3: 'time'},
                                'output': {0: 'batch_size'}})

onnx.checker.check_model("audio_model.onnx")
session = ort.InferenceSession("audio_model.onnx", providers=["CPUExecutionProvider"])
smoke_output = session.run(None, {session.get_inputs()[0].name: dummy_input.numpy()})[0]
assert smoke_output.shape == (1, len(CLASS_NAMES))

artifacts_dir = Path("/kaggle/working/artifacts")
artifacts_dir.mkdir(exist_ok=True)
shutil.copy2("audio_model.onnx", artifacts_dir / "audio_model.onnx")
shutil.copy2("audio_labels.json", artifacts_dir / "audio_labels.json")
shutil.copy2(generated_root / "dataset_manifest.json", artifacts_dir / "audio_dataset_manifest.json")
print(f"Audio ONNX smoke test passed. Download artifacts from: {artifacts_dir}")